In [11]:
from tkinter import *
from tkinter import ttk
from collections import deque
from queue import PriorityQueue
import random
import copy
def zero(state):

    for i in range(3):
        for j in range(3):

            if state[i][j] == 0:
                return i, j


def tap_con(state):

    x, y = zero(state)

    moves = []

    if x > 0:
        moves.append((x - 1, y))

    if x < 2:
        moves.append((x + 1, y))

    if y > 0:
        moves.append((x, y - 1))

    if y < 2:
        moves.append((x, y + 1))

    result = []

    for nx, ny in moves:

        new = copy.deepcopy(state)

        new[x][y], new[nx][ny] = \
        new[nx][ny], new[x][y]

        result.append(new)

    return result


def random_goal():

    nums = list(range(9))
    random.shuffle(nums)

    return [
        nums[:3],
        nums[3:6],
        nums[6:]
    ]


goal = random_goal()

start = copy.deepcopy(goal)

for _ in range(20):

    start = random.choice(tap_con(start))


# BFS

def bfs(start):

    frontier = deque([[start]])

    reached = set()

    nodes = 0

    while frontier:

        path = frontier.popleft()

        state = path[-1]

        reached.add(str(state))

        for con in tap_con(state):

            nodes += 1

            if con == goal:

                return path + [con], nodes

            if (str(con) not in reached and
                con not in [p[-1] for p in frontier]):

                frontier.append(path + [con])

    return None, nodes


# DFS
def dfs(start):

    stack = [[start]]

    reached = set()

    nodes = 0

    while stack:

        path = stack.pop()

        state = path[-1]

        reached.add(str(state))

        for con in tap_con(state):

            nodes += 1

            if con == goal:

                return path + [con], nodes

            if (str(con) not in reached and
                con not in [p[-1] for p in stack]):

                stack.append(path + [con])

    return None, nodes


# IDS
def dls(start, limit):

    stack = [[start]]

    reached = set()

    nodes = 0

    while stack:

        path = stack.pop()

        state = path[-1]

        reached.add(str(state))

        # độ sâu hiện tại
        depth = len(path) - 1

        if state == goal:

            return path, nodes

        # đạt giới hạn độ sâu
        if depth >= limit:
            continue

        for con in tap_con(state):

            nodes += 1

            if (str(con) not in reached and
                con not in [p[-1] for p in stack]):

                stack.append(path + [con])

    return None, nodes


def ids(start):

    total_nodes = 0

    depth = 0

    while True:

        path, nodes = dls(start, depth)

        total_nodes += nodes

        if path:

            return path, total_nodes

        depth += 1

#UCS
# UCS
def cost(state):

    wrong = 0

    for i in range(3):
        for j in range(3):

            if state[i][j] != goal[i][j]:
                wrong += 1

    return wrong


def ucs(start):

    frontier = PriorityQueue()

    nodes = 0

    count = 0

    # (priority, count, path)
    frontier.put((cost(start), count, [start]))

    reached = set()

    while not frontier.empty():

        current_cost, _, path = frontier.get()

        state = path[-1]

        if state == goal:

            return path, nodes

        reached.add(str(state))

        for con in tap_con(state):

            nodes += 1

            if (str(con) not in reached and
                con not in [p[2][-1] for p in frontier.queue]):

                count += 1

                new_cost = cost(con)

                frontier.put(
                    (
                        new_cost,
                        count,
                        path + [con]
                    )
                )

    return None, nodes

# GREEDY SEARCH
# h(n) = manhattan

def heuristic(state):

    distance = 0

    for num in range(1, 9):

        for i in range(3):
            for j in range(3):

                if state[i][j] == num:
                    x1, y1 = i, j

                if goal[i][j] == num:
                    x2, y2 = i, j

        distance += abs(x1 - x2) + abs(y1 - y2)

    return distance


def greedy(start):

    frontier = PriorityQueue()

    reached = set()

    nodes = 0

    count = 0

    # (heuristic, count, path)
    frontier.put(
        (
            heuristic(start),
            count,
            [start]
        )
    )

    while not frontier.empty():

        _, _, path = frontier.get()

        state = path[-1]

        # gặp goal
        if state == goal:

            return path, nodes

        reached.add(str(state))

        # xét các trạng thái con
        for con in tap_con(state):

            nodes += 1

            if (str(con) not in reached and
                con not in [p[2][-1] for p in frontier.queue]):

                count += 1

                frontier.put(
                    (
                        heuristic(con),
                        count,
                        path + [con]
                    )
                )

    return None, nodes

# A*    g(n): số bước đã đi h(n):manhattan
def astar(start):

    frontier = PriorityQueue()

    reached = set()

    nodes = 0

    count = 0

    # g(start) = 0
    g = 0

    # f = g + h
    f = g + heuristic(start)

    # (f, count, g, path)
    frontier.put(
        (
            f,
            count,
            g,
            [start]
        )
    )

    while not frontier.empty():

        _, _, g, path = frontier.get()

        state = path[-1]

        if state == goal:

            return path, nodes

        reached.add(str(state))

        for con in tap_con(state):

            nodes += 1

            if (str(con) not in reached and
                con not in [p[3][-1] for p in frontier.queue]):

                count += 1

                new_g = g + 1

                new_f = new_g + heuristic(con)

                frontier.put(
                    (
                        new_f,
                        count,
                        new_g,
                        path + [con]
                    )
                )

    return None, nodes

# IDA*  g(n): số bước đã đi h(n):manhattan
def ida_search(path, g, threshold):

    global ida_nodes

    state = path[-1]

    f = g + heuristic(state)

    # vượt ngưỡng
    if f > threshold:

        return f

    # gặp goal
    if state == goal:

        return path

    minimum = float("inf")

    for con in tap_con(state):

        ida_nodes += 1

        # tránh lặp trong path hiện tại
        if con not in path:

            result = ida_search(
                path + [con],
                g + 1,
                threshold
            )

            # tìm thấy lời giải
            if isinstance(result, list):

                return result

            # lưu f nhỏ nhất bị vượt
            if result < minimum:

                minimum = result

    return minimum

def ida_star(start):

    global ida_nodes

    ida_nodes = 0

    threshold = heuristic(start)

    while True:

        result = ida_search(
            [start],
            0,
            threshold
        )

        # tìm thấy đường đi
        if isinstance(result, list):

            return result, ida_nodes

        # thất bại
        if result == float("inf"):

            return None, ida_nodes

        # tăng threshold mới
        threshold = result

# leo doi don gian
# dùng Manhattan
def hill_climbing(start):

    current = start

    path = [current]

    nodes = 0

    while True:

        current_value = heuristic(current)

        found_better = False

        for neighbor in tap_con(current):

            nodes += 1

            neighbor_value = heuristic(neighbor)

            # tốt hơn -> đi luôn
            if neighbor_value < current_value:

                current = neighbor

                path.append(current)

                found_better = True

                break

        # gặp goal
        if current == goal:

            return path, nodes

        # không có trạng thái nào tốt hơn
        if not found_better:

            return path, nodes

# leo doi doc nhat
def steepest_climbing(start):

    current = start

    path = [current]

    nodes = 0

    while True:

        current_value = heuristic(current)

        neighbors = tap_con(current)

        if not neighbors:
            return path, nodes

        best_neighbor = None
        best_value = current_value

        # duyệt toàn bộ trạng thái lân cận
        for neighbor in neighbors:

            nodes += 1

            value = heuristic(neighbor)

            if value < best_value:

                best_value = value
                best_neighbor = neighbor

        # không có trạng thái nào tốt hơn
        if best_neighbor is None:
            return path, nodes

        current = best_neighbor

        path.append(current)

        if current == goal:
            return path, nodes

#GUI
root = Tk()

root.title("8 Puzzle Search")
root.geometry("1200x700")
root.config(bg="#f2f2f2")


# Frames

left_frame = Frame(root, bg="white", padx=30, pady=20)
left_frame.pack(side=LEFT, fill=Y, padx=15, pady=15)

middle_frame = Frame(root, bg="white", padx=30, pady=20)
middle_frame.pack(side=LEFT, fill=Y, pady=15)

right_frame = Frame(root, bg="white", padx=20, pady=20)
right_frame.pack(side=LEFT, fill=BOTH, expand=True, padx=15, pady=15)


# Create board
def create_board(parent, size=28):

    frame = Frame(parent, bg="white")

    board = []

    for i in range(3):

        row = []

        for j in range(3):

            cell = Label(
                frame,
                text="",
                width=3,
                height=1,
                font=("Arial", size, "bold"),
                relief="solid",
                borderwidth=2,
                bg="white"
            )

            cell.grid(row=i, column=j)

            row.append(cell)

        board.append(row)

    return frame, board


def draw(board, state):

    for i in range(3):
        for j in range(3):

            value = state[i][j]

            if value == 0:
                board[i][j]["text"] = ""
            else:
                board[i][j]["text"] = str(value)

# Start
Label(
    left_frame,
    text="Start",
    font=("Arial", 32, "bold"),
    bg="white"
).pack(pady=10)

start_frame, start_board = create_board(
    left_frame,
    size=18
)

start_frame.pack()


# Goal
Label(
    left_frame,
    text="Goal",
    font=("Arial", 32, "bold"),
    bg="white"
).pack(pady=35)

goal_frame, goal_board = create_board(
    left_frame,
    size=18
)
goal_frame.pack()

draw(start_board, start)
draw(goal_board, goal)

#lua chon thuat toan
Label(
    middle_frame,
    text="Algorithm",
    font=("Arial", 32, "bold"),
    bg="white"
).pack(pady=10)

algo = StringVar(value="BFS")
algo_frame = Frame(
    middle_frame,
    bg="white"
)

algo_frame.pack(pady=10)

algorithms = [
    "BFS",
    "DFS",
    "IDS",
    "UCS",
    "GREEDY",
    "A*",
    "IDA*",
    "HILL",
    "STEEPEST"
]

for index, name in enumerate(algorithms):

    row = index // 3
    col = index % 3

    Radiobutton(
        algo_frame,
        text=name,
        variable=algo,
        value=name,
        indicatoron=False,
        width=10,
        height=2,
        font=("Arial", 16, "bold"),
        bg="white"
    ).grid(
        row=row,
        column=col,
        padx=10,
        pady=10
    )

def animate(path, step=0):

    if step < len(path):

        draw(start_board, path[step])

        root.after(
            500,
            animate,
            path,
            step + 1
        )

# Buttons
def solve():

    if algo.get() == "BFS":

        path, nodes = bfs(start)

    elif algo.get() == "DFS":

        path, nodes = dfs(start)

    elif algo.get() == "IDS":

        path, nodes = ids(start)

    elif algo.get() == "UCS":

        path, nodes = ucs(start)

    elif algo.get() == "GREEDY":
        path, nodes = greedy(start)

    elif algo.get() == "A*":
        path, nodes = astar(start)

    elif algo.get() == "IDA*":
        path, nodes = ida_star(start)

    elif algo.get() == "HILL":
        path, nodes = hill_climbing(start)

    elif algo.get() == "STEEPEST":
        path, nodes = steepest_climbing(start)



    if path and path[-1] == goal:

        animate(path)

        show_states(path)

        info_label.config(
            text=f"Steps: {len(path)-1}\nNodes: {nodes}"
        )

    else:

        animate(path)

        show_states(path)

        info_label.config(
            text=f"Local Optimum\nSteps: {len(path)-1}\nNodes: {nodes}"
        )


Button(
    middle_frame,
    text="Solve",
    command=solve,
    width=12,
    height=2,
    bg="#2f80ed",
    fg="white",
    font=("Arial", 18, "bold")
).pack(pady=25)


def new_puzzle():

    global start
    global goal

    goal = random_goal()

    start = copy.deepcopy(goal)

    for _ in range(20):

        start = random.choice(tap_con(start))

    draw(start_board, start)
    draw(goal_board, goal)

    info_label.config(text="Steps: 0\nNodes: 0")

    for widget in states_frame.winfo_children():
        widget.destroy()


Button(
    middle_frame,
    text="New Puzzle",
    command=new_puzzle,
    width=12,
    height=2,
    font=("Arial", 14, "bold")
).pack(pady=5)


# Info
info_label = Label(
    middle_frame,
    text="Steps: 0\nNodes: 0",
    font=("Arial", 18),
    bg="white",
    justify=LEFT
)

info_label.pack(side=BOTTOM, pady=20)


# States
Label(
    right_frame,
    text="States",
    font=("Arial", 32, "bold"),
    bg="white"
).pack(pady=10)

canvas = Canvas(
    right_frame,
    bg="white",
    highlightthickness=0
)

scrollbar = ttk.Scrollbar(
    right_frame,
    orient="vertical",
    command=canvas.yview
)

canvas.configure(yscrollcommand=scrollbar.set)

scrollbar.pack(side=RIGHT, fill=Y)

canvas.pack(fill=BOTH, expand=True)

states_frame = Frame(canvas, bg="white")

canvas.create_window(
    (0, 0),
    window=states_frame,
    anchor="nw"
)

states_frame.bind(
    "<Configure>",
    lambda e: canvas.configure(
        scrollregion=canvas.bbox("all")
    )
)



def show_states(path):

    for widget in states_frame.winfo_children():
        widget.destroy()

    cols = 3

    for index, state in enumerate(path):

        row = index // cols
        col = index % cols

        container = Frame(
            states_frame,
            bg="white"
        )

        container.grid(
            row=row,
            column=col,
            padx=35,
            pady=20
        )

        mini_frame, mini_board = create_board(
            container,
            size=14
        )

        for i in range(3):
            for j in range(3):

                mini_board[i][j].config(
                    width=2,
                    height=1
                )

        draw(mini_board, state)

        mini_frame.pack()


root.mainloop()